# Trimorious 2.0 | M2.0a — Load pinned Qwen3 in 4-bit and run 10-step smoke test

**Notebook 03 of 07 · GPU runtime**

**Use a clean T4 GPU runtime.** Notebook 02 must have produced a valid dataset approval. This notebook loads Qwen3-4B-Instruct-2507 at the pinned revision with 4-bit NF4, double quantization, FP16, LoRA rank 8, gradient checkpointing and assistant-only labels. It checks for truncation, finite loss and adapter gradients. Runs ten optimizer steps only, writes `smoke_pass.json` and exits the GPU worker, releasing its CUDA context. The smoke adapter is not used for full training. On failure, inspect the error and do not run notebook 04. If settings are changed, use exactly the same changed settings in notebooks 03 and 04 and rerun smoke. No model upload.

All stages use the same `MyDrive/trimorious_PRIVATE_m1_4_1/release_v2/` workspace. Save this notebook in Drive. Edit only the configuration and select Runtime → Run all. Never run another GPU notebook concurrently. Each notebook is self-contained and contains no credentials. GPU execution is not locally tested.

In [ ]:
# @title Configuration — SMOKE
STAGE = 'SMOKE'
USE_DRIVE = True # @param {type:'boolean'}
INSTALL_GPU_DEPS = True # @param {type:'boolean'}
PROJECT_ROOT = '/content/drive/MyDrive/trimorious_PRIVATE_m1_4_1'
SOURCE_NAME = 'trimorious_m1_4_1_PRIVATE_GPU_partial_20261127T061605946898Z.zip'
D07_FILENAME = 'trimorious_m1_3_2_PRIVATE_full_1c3d70c27cc3c9f8.zip'
# Save this benchmark ZIP in PROJECT_ROOT (or upload to /content once). Required for EVALUATE.
FRESH_TEST_PATH = '' # @param {type:'string'}
# Fresh test JSONL format documented in section 5. Must be externally authored, sealed before evaluation; preferably before training.
FRESH_TEST_SHA256 = '' # @param {type:'string'}
FRESH_INDEPENDENCE_ATTESTED = False # @param {type:'boolean'}
# Confirm external authorship, expert key review, no access to this adapter's evaluation outputs, no tuning on the test.
DATA_LICENSE_REVIEWED = False # @param {type:'boolean'}
DATA_REVIEW_APPROVED = False # @param {type:'boolean'}
# By approving, you confirm rights to train and distribute the resulting adapter, and actual review of CSV rows.
HUMAN_EVAL_APPROVED = False # @param {type:'boolean'}
# After actual rubric review; see evaluation report. Not an automatic pass.
PUBLISH_APPROVAL = '' # @param {type:'string'}
MAKE_PUBLIC = False # @param {type:'boolean'}
PUBLIC_APPROVAL = '' # @param {type:'string'}
MAX_TRAIN_PER_TEMPLATE = 12
MAX_VAL_PER_TEMPLATE = 6
MAX_LENGTH = 512
GRAD_ACCUM = 16
EPOCHS = 1
SEED = 417031
# Provisional minimum release policy: approve before training; cannot change after dataset freeze.
MIN_D07_GAIN = 1.0       # percentage points, measurable diagnostic gain
MAX_FAMILY_DROP = 5.0   # percentage points
MIN_FRESH_GAIN = 0.0
MAX_PROTOCOL_DROP = 2.0
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
REV = 'cdbee75f17c01a7cc42f958dc650907174af0554'
TARGET = 'roime81/trimorious-research-4B-v2'
EXPECTED_CORPUS = '9fe5e9dbce17096c4d1ddd28a82e45111a37cbba66c574ec63473f6f70dd283a'

In [ ]:
# @title Mount Drive and prepare local worker scripts
from pathlib import Path
import json, os, sys, subprocess, hashlib, csv, sqlite3, shutil, traceback
from collections import Counter
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT = Path(PROJECT_ROOT)
assert ROOT.is_dir(), f'Project folder missing: {ROOT}'
WORK = ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
SCRIPT_DIR = Path('/tmp/trimorious_v2_sequence'); SCRIPT_DIR.mkdir(exist_ok=True)
print('Persistent workspace:', WORK)
print('Local script directory:', SCRIPT_DIR)
CONFIG_NAMES=['USE_DRIVE', 'INSTALL_GPU_DEPS', 'PROJECT_ROOT', 'SOURCE_NAME', 'D07_FILENAME', 'FRESH_TEST_PATH', 'FRESH_TEST_SHA256', 'FRESH_INDEPENDENCE_ATTESTED', 'DATA_LICENSE_REVIEWED', 'DATA_REVIEW_APPROVED', 'HUMAN_EVAL_APPROVED', 'PUBLISH_APPROVAL', 'MAKE_PUBLIC', 'PUBLIC_APPROVAL', 'MAX_TRAIN_PER_TEMPLATE', 'MAX_VAL_PER_TEMPLATE', 'MAX_LENGTH', 'GRAD_ACCUM', 'EPOCHS', 'SEED', 'MIN_D07_GAIN', 'MAX_FAMILY_DROP', 'MIN_FRESH_GAIN', 'MAX_PROTOCOL_DROP', 'BASE', 'REV', 'TARGET', 'EXPECTED_CORPUS']
CONFIG={k:globals()[k] for k in CONFIG_NAMES}
assert TARGET=='roime81/trimorious-research-4B-v2' and BASE=='Qwen/Qwen3-4B-Instruct-2507' and REV=='cdbee75f17c01a7cc42f958dc650907174af0554'

## Embedded implementation
The following cells write the implementation to the local runtime. Run them unchanged. No source code download required.

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/common.py
# @title 1. Mount Drive and common helpers (CPU)
import os, sys, json, hashlib, math, re, random, csv, zipfile, sqlite3, statistics, time, gc, shutil, platform
from pathlib import Path
from collections import Counter, defaultdict
from fractions import Fraction
from importlib import metadata
assert STAGE in {'RECOVER','APPROVE','SMOKE','TRAIN','EVALUATE','PACKAGE','PUBLISH'}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT=Path(PROJECT_ROOT); assert ROOT.is_dir(), f'Missing project folder: {ROOT}'
WORK=ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False)
def sha(b): return hashlib.sha256(b if isinstance(b,bytes) else canon(b).encode()).hexdigest()
def put(p,obj):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False)); tmp.replace(p)
def get(p): return json.loads(Path(p).read_text())
def lines(b): return [json.loads(x) for x in b.splitlines() if x.strip()]
def readl(p): return lines(Path(p).read_text())
def writel(p,rows):
    p=Path(p); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(''.join(canon(r)+'\n' for r in rows)); tmp.replace(p)
def writecsv(p,rows,fields):
    with Path(p).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
def archive_check(z):
    ns=z.namelist(); assert len(ns)==len(set(ns)) and z.testzip() is None
    m=json.loads(z.read('archive_manifest.json')); assert set(m)==set(ns)-{'archive_manifest.json'}
    assert all(sha(z.read(k))==v for k,v in m.items()), 'Archive hash mismatch'
    return m
def state(stage,**kw):
    p=WORK/'project_state.json'; s=get(p) if p.exists() else {}; s[stage]=kw; put(p,s)
def norm(s): return re.sub(r'\s+',' ',s.casefold()).strip()
def mask(s): return re.sub(r'(?<![a-z])[-+]?\d+(?:\.\d+)?(?:e[-+]?\d+)?','NUM',norm(s))
def close(a,b): return math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-10)
POLICY={'min_d07_gain_pp':MIN_D07_GAIN,'max_family_drop_pp':MAX_FAMILY_DROP,'min_fresh_gain_pp':MIN_FRESH_GAIN,'max_protocol_drop_pp':MAX_PROTOCOL_DROP}
print('Stage:',STAGE,'| Persistent work directory:',WORK)

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/approve.py
def load_approved():
    approval=get(WORK/'dataset_approval.json'); assert approval['training_approved'] is True
    for name,digest in approval['files'].items(): assert sha((WORK/name).read_bytes())==digest, f'Changed approved file: {name}'
    assert approval['policy']==POLICY and approval['base']==BASE and approval['revision']==REV
    return approval

if STAGE=='APPROVE':
    assert DATA_LICENSE_REVIEWED and DATA_REVIEW_APPROVED, 'Review rights and examples first; then set both approval flags.'
    report=get(WORK/'recovery.json'); candidates=readl(WORK/'review_candidates.jsonl'); assert sha(candidates)==report['review_set_sha256']
    with (WORK/'review_queue.csv').open(newline='',encoding='utf-8-sig') as f: reviews=list(csv.DictReader(f))
    by_id={r['id']:r for r in reviews}; assert len(reviews)==len(by_id)==len(candidates) and set(by_id)=={r['id'] for r in candidates}
    approved={'train':[],'validation':[]}; dispositions=[]
    for r in candidates:
        q=by_id[r['id']]; assert q['prompt']==r['prompt'] and q['response']==r['reference_response'] and q['split']==r['split'] and q['template']==r['template_id']
        d=q['decision'].strip().lower(); assert d in {'','approve','reject'}
        if d: assert q['reviewer'].strip() and q['notes'].strip(), 'Each explicit decision requires reviewer and rationale.'
        dispositions.append({'id':r['id'],'decision':d or 'unreviewed','reviewer':q['reviewer'],'notes':q['notes']})
        if d=='approve':
            assert close(recompute(r),r['numeric_answer']) and not r['d07_screen']['flagged']
            # Use the source response as written, not unapproved teacher output.
            split='train' if r['split']=='development_candidate' else 'validation'
            approved[split].append({'id':r['id'],'template':r['template_id'],'family':r['family'],'messages':[{'role':'user','content':r['prompt']},{'role':'assistant','content':r['reference_response']}]})
    assert len(approved['train'])>=32 and len(approved['validation'])>=8, 'Review at least 32 development and 8 validation examples; these are smoke-level minima, not a quality guarantee.'
    assert {r['template'] for r in approved['train']}.isdisjoint({r['template'] for r in approved['validation']})
    assert not ({mask(r['messages'][0]['content']) for r in approved['train']} & {mask(r['messages'][0]['content']) for r in approved['validation']})
    if FRESH_TEST_SHA256: assert re.fullmatch('[a-f0-9]{64}',FRESH_TEST_SHA256)
    data_names=['train.jsonl','validation.jsonl','review_dispositions.jsonl']
    contents=[approved['train'],approved['validation'],dispositions]
    files={name:sha(''.join(canon(r)+'\n' for r in rows).encode()) for name,rows in zip(data_names,contents)}
    approval={'training_approved':True,'license_review_attested':True,'corpus_sha256':EXPECTED_CORPUS,'review_set_sha256':report['review_set_sha256'],'files':files,'counts':{k:len(v) for k,v in approved.items()},'policy':POLICY,'fresh_test_sha256':FRESH_TEST_SHA256,'base':BASE,'revision':REV,'scope':'human-reviewed, recomputed synthetic math/science/statistics; no teacher or tool trace SFT'}
    if (WORK/'dataset_approval.json').exists(): assert get(WORK/'dataset_approval.json')==approval, 'Approval already frozen. New revision requires separate workspace.'
    for name,rows in zip(data_names,contents): writel(WORK/name,rows)
    put(WORK/'dataset_approval.json',approval); state('APPROVE',complete=True,counts=approval['counts']); print(json.dumps(approval,indent=2)); print('NEXT: SMOKE')

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/gpu.py
if STAGE in {'SMOKE','TRAIN','EVALUATE'}:
    import subprocess
    if INSTALL_GPU_DEPS:
        lock_path=WORK/'auto_gpu_versions.json'
        if lock_path.exists():
            locked=get(lock_path)
            assert metadata.version('torch')==locked['torch'], 'PyTorch changed in this runtime. Use a compatible runtime; do not mix training checkpoints.'
            specs=[k+'=='+v for k,v in locked.items() if k!='torch']+['tqdm','scipy','scikit-learn']
        else:
            specs=['transformers==4.57.6','peft>=0.17,<0.19','accelerate>=1.6,<2','bitsandbytes>=0.48,<1','huggingface_hub>=0.34,<1','safetensors>=0.5','tqdm','scipy','scikit-learn']
        subprocess.check_call([sys.executable,'-m','pip','install','-q',*specs])
    import torch
    from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig,Trainer,TrainingArguments,TrainerCallback,set_seed
    from transformers.trainer_utils import get_last_checkpoint
    from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel
    from tqdm.auto import tqdm
    assert torch.cuda.is_available(), 'Select a GPU runtime.'
    versions={x:metadata.version(x) for x in ['torch','transformers','peft','accelerate','bitsandbytes','huggingface_hub','safetensors']}
    lock_path=WORK/'auto_gpu_versions.json'
    if lock_path.exists(): assert get(lock_path)==versions, 'GPU dependency versions changed; stop rather than mix checkpoints.'
    else: put(lock_path,versions)
    print('GPU:',torch.cuda.get_device_name(0),'Versions:',versions,flush=True)
    def load_base(train=False):
        gc.collect(); torch.cuda.empty_cache(); free,total=torch.cuda.mem_get_info()
        print(f'Free GPU: {free/2**30:.2f}/{total/2**30:.2f} GiB',flush=True)
        assert free/2**30>=10, 'Not enough free GPU memory. Use a clean runtime; do not load teacher models.'
        tok=AutoTokenizer.from_pretrained(BASE,revision=REV,trust_remote_code=False)
        assert tok.chat_template
        if tok.pad_token_id is None: tok.pad_token=tok.eos_token
        tok.padding_side='right'
        quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
        mdl=AutoModelForCausalLM.from_pretrained(BASE,revision=REV,quantization_config=quant,device_map={'':0},torch_dtype=torch.float16,attn_implementation='eager',low_cpu_mem_usage=True,trust_remote_code=False,use_safetensors=True)
        assert getattr(mdl,'is_loaded_in_4bit',False)
        mdl.config.use_cache=not train
        if train:
            mdl=prepare_model_for_kbit_training(mdl,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
            mdl=get_peft_model(mdl,LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],revision=REV))
            mdl.print_trainable_parameters(); assert all('lora_' in name for name,p in mdl.named_parameters() if p.requires_grad)
        return mdl,tok
    class Rows(torch.utils.data.Dataset):
        def __init__(self,data): self.data=data
        def __len__(self): return len(self.data)
        def __getitem__(self,i): return self.data[i]
    def tokenize_rows(rows,tok):
        result=[]; too_long=[]
        for r in rows:
            msgs=r['messages']; prefix=tok.apply_chat_template(msgs[:-1],tokenize=True,add_generation_prompt=True); full=tok.apply_chat_template(msgs,tokenize=True,add_generation_prompt=False)
            assert full[:len(prefix)]==prefix, f'Chat template prefix mismatch: {r["id"]}'
            assert len(full)>len(prefix) and tok.eos_token_id in full[len(prefix):], f'No supervised EOS: {r["id"]}'
            if len(full)>MAX_LENGTH: too_long.append(r['id']); continue
            labels=[-100]*len(prefix)+full[len(prefix):]; assert all(x==-100 for x in labels[:len(prefix)]) and any(x!=-100 for x in labels[len(prefix):])
            result.append({'input_ids':full,'attention_mask':[1]*len(full),'labels':labels})
        assert not too_long, f'Would truncate {len(too_long)} answers; revise config before smoke: {too_long[:5]}'
        print('Tokenized',len(result),'max length',max(len(r['input_ids']) for r in result)); return Rows(result)
    def collate(items):
        n=max(len(r['input_ids']) for r in items); out={}
        for key,pad in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]: out[key]=torch.tensor([r[key]+[pad]*(n-len(r[key])) for r in items],dtype=torch.long)
        return out
    class Health(TrainerCallback):
        def __init__(self): self.nonzero=False; self.logs=[]
        def on_pre_optimizer_step(self,args,state,control,model=None,**kw):
            grads=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
            assert grads and all(torch.isfinite(g).all().item() for g in grads), 'Missing or nonfinite adapter gradients'
            self.nonzero |= any(torch.count_nonzero(g).item()>0 for g in grads)
        def on_log(self,args,state,control,logs=None,**kw):
            if logs:
                for k in ('loss','eval_loss','train_loss','grad_norm'):
                    if k in logs: assert math.isfinite(float(logs[k])),f'Nonfinite {k}'
                row={**logs,'step':state.global_step,'peak_gpu_gib':torch.cuda.max_memory_allocated()/2**30}; self.logs.append(row); print(row,flush=True)
    def training_config(approval): return {'approval_sha256':sha(approval),'base':BASE,'revision':REV,'max_length':MAX_LENGTH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'seed':SEED,'lora_rank':8,'alpha':16,'lr':1e-4,'precision':'fp16_nf4','versions':versions}

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/train.py
if STAGE in {'SMOKE','TRAIN'}:
    approval=load_approved(); cfg=training_config(approval); fingerprint=sha(cfg)[:20]
    run_dir=WORK/'sft_output'/fingerprint; run_dir.mkdir(parents=True,exist_ok=True)
    if (run_dir/'config.json').exists(): assert get(run_dir/'config.json')==cfg
    put(run_dir/'config.json',cfg)
    skip_run=False
    if STAGE=='SMOKE' and (run_dir/'smoke_pass.json').exists():
        smoke=get(run_dir/'smoke_pass.json'); assert smoke['passed'] is True and smoke['config']==cfg and smoke['steps']==10
        print('Matching smoke test already passed; reusing.',flush=True); skip_run=True
    if STAGE=='TRAIN' and (run_dir/'training_complete.json').exists():
        done=get(run_dir/'training_complete.json'); assert done['config']==cfg
        assert all(sha((run_dir/'final_adapter'/k).read_bytes())==v for k,v in done['adapter_hashes'].items())
        put(WORK/'active_run.json',{'run':run_dir.name}); state('TRAIN',complete=True,run=str(run_dir)); print('Matching trained adapter already saved; reusing.',flush=True); skip_run=True
    if not skip_run:
        if STAGE=='TRAIN':
            smoke=get(run_dir/'smoke_pass.json'); assert smoke['passed'] is True and smoke['config']==cfg, 'Run matching SMOKE stage first.'
            assert not (run_dir/'training_complete.json').exists(), 'Training already completed; use EVALUATE.'
        set_seed(SEED); torch.cuda.reset_peak_memory_stats()
        model,tokenizer=load_base(train=True)
        tr=tokenize_rows(readl(WORK/'train.jsonl'),tokenizer); va=tokenize_rows(readl(WORK/'validation.jsonl'),tokenizer)
        example=collate([tr[0]]); assert (example['labels']==-100).any() and (example['labels']!=-100).any()
        health=Health(); smoke_mode=STAGE=='SMOKE'
        output=run_dir/('smoke_scratch' if smoke_mode else 'checkpoints'); output.mkdir(exist_ok=True)
        args=TrainingArguments(output_dir=str(output),per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=GRAD_ACCUM,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},fp16=True,bf16=False,optim='paged_adamw_8bit',learning_rate=1e-4,max_grad_norm=1.0,warmup_ratio=.03,num_train_epochs=EPOCHS,max_steps=10 if smoke_mode else -1,logging_steps=1,logging_nan_inf_filter=False,eval_strategy='no' if smoke_mode else 'epoch',save_strategy='no' if smoke_mode else 'steps',save_steps=5,save_total_limit=2,save_safetensors=True,report_to='none',seed=SEED,data_seed=SEED,remove_unused_columns=False,prediction_loss_only=True,disable_tqdm=False)
        trainer=Trainer(model=model,args=args,train_dataset=tr,eval_dataset=va,data_collator=collate,callbacks=[health])
        resume=None if smoke_mode else get_last_checkpoint(str(output))
        if resume:
            p=Path(resume); assert all((p/f).is_file() for f in ['adapter_model.safetensors','adapter_config.json','trainer_state.json','optimizer.pt','scheduler.pt','rng_state.pth']), 'Latest checkpoint incomplete; inspect earlier checkpoint before resuming.'
        print('Resume:',resume,'| training rows:',len(tr),'| validation rows:',len(va),flush=True)
        result=trainer.train(resume_from_checkpoint=resume)
        assert math.isfinite(result.training_loss) and health.nonzero, 'No verified nonzero adapter gradients or invalid loss'
        if smoke_mode:
            assert trainer.state.global_step==10
            put(run_dir/'smoke_pass.json',{'passed':True,'config':cfg,'steps':10,'loss':result.training_loss,'logs':health.logs}); state('SMOKE',complete=True,run=str(run_dir)); print('SMOKE PASSED. Open notebook 04 in a fresh GPU runtime.')
        else:
            metrics=trainer.evaluate(); assert math.isfinite(metrics['eval_loss'])
            final=run_dir/'final_adapter'; final.mkdir(exist_ok=True); trainer.save_model(str(final)); tokenizer.save_pretrained(str(final))
            assert (final/'adapter_model.safetensors').is_file()
            config=get(final/'adapter_config.json'); assert config['base_model_name_or_path']==BASE and config.get('revision')==REV
            hashes={p.name:sha(p.read_bytes()) for p in final.iterdir() if p.is_file()}
            put(run_dir/'training_complete.json',{'config':cfg,'metrics':metrics,'steps':trainer.state.global_step,'train_loss':result.training_loss,'logs':health.logs,'adapter_hashes':hashes}); put(WORK/'active_run.json',{'run':run_dir.name}); state('TRAIN',complete=True,run=str(run_dir)); print('TRAINING COMPLETE. Open notebook 05 when evaluation inputs are ready.')
        del trainer,model,tokenizer; gc.collect(); torch.cuda.empty_cache()

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/worker.py
import sys, json
from pathlib import Path
SCRIPTS=Path(__file__).resolve().parent
stage=sys.argv[2]
assert stage in {'SMOKE','TRAIN','EVALUATE'}
ns=json.loads(Path(sys.argv[1]).read_text())
ns.update(STAGE=stage,USE_DRIVE=False,__name__='__main__')
files=['common','approve','gpu']+(['train'] if stage in {'SMOKE','TRAIN'} else ['eval_setup','scorer','evaluate'])
for name in files:
    p=SCRIPTS/(name+'.py'); exec(compile(p.read_text(),str(p),'exec'),ns)


## Execute stage
Read the actual outputs. A stopped or failed gate is not a completed stage.

In [ ]:
# @title Execute this notebook stage
def file_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def read_json(p): return json.loads(Path(p).read_text())
def save_json(p,x):
    p=Path(p); temp=p.with_suffix(p.suffix+'.tmp'); temp.write_text(json.dumps(x,indent=2)); temp.replace(p)
def record_controller(stage,status,detail):
    save_json(WORK/'auto_controller_state.json',{'stage':stage,'status':status,'detail':str(detail)})
def pause(stage,detail):
    record_controller(stage,'waiting',detail); print('\nPAUSED — '+stage+'\n'+str(detail)+'\nSave changes, then Runtime → Run all. Existing artifacts are preserved.',flush=True)
def cpu(stage, parts):
    ns={**CONFIG,'USE_DRIVE':False,'STAGE':stage,'__name__':'__main__'}
    for part in ['common']+parts:
        p=SCRIPT_DIR/(part+'.py'); exec(compile(p.read_text(),str(p),'exec'),ns)
    return ns

def gpu(stage):
    print('\n=== '+stage+' — isolated GPU worker ===',flush=True)
    record_controller(stage,'running','GPU subprocess')
    config_path=SCRIPT_DIR/'run_config.json'; save_json(config_path,CONFIG)
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
    p=subprocess.Popen([sys.executable,'-u',str(SCRIPT_DIR/'worker.py'),str(config_path),stage],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout:
            print(line,end='',flush=True)
        status=p.wait()
    except BaseException:
        p.terminate()
        try: p.wait(timeout=20)
        except subprocess.TimeoutExpired: p.kill(); p.wait()
        raise
    if status: raise RuntimeError(f'{stage} worker exited with code {status}. Read the actual error above; checkpoints preserved.')
    # Child exit releases CUDA allocations. Dependencies already installed for subsequent child processes in this session.
    CONFIG['INSTALL_GPU_DEPS']=False

def validate_active(ns):
    approval,run,done,final=ns['active_run']()
    cfg=done['config']
    for k,v in {'base':BASE,'revision':REV,'max_length':MAX_LENGTH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'seed':SEED}.items():
        assert cfg[k]==v,f'Trained adapter uses different {k}; do not silently change a finished run.'
    assert done['steps']>0 and (final/'adapter_model.safetensors').is_file()
    return approval,run,done,final

def get_eval_artifacts(run):
    active=read_json(WORK/'active_evaluation.json'); assert active['run']==run.name
    folder=run/'evaluation'/active['evaluation']; return folder,read_json(folder/'evaluation_report.json')

def eval_receipt(run,done):
    folder,report=get_eval_artifacts(run)
    assert report['evaluation_config']['adapter_hashes']==done['adapter_hashes']
    assert report['evaluation_config']['fresh_hash']==FRESH_TEST_SHA256 and report['fresh']
    db=folder/'responses.sqlite3'
    with sqlite3.connect(db.resolve().as_uri()+'?mode=ro',uri=True) as conn:
        assert conn.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        counts=dict(conn.execute('SELECT model,COUNT(*) FROM responses GROUP BY model'))
        expected=1000+report['fresh']['n']
        assert counts=={'qwen3_base':expected,'trimorious_v2':expected}, f'Incomplete paired evaluation: {counts}'
    return {'evaluation':folder.name,'report_sha256':file_sha(folder/'evaluation_report.json'),'db_sha256':file_sha(db),'mapping_sha256':file_sha(folder/'rubric_mapping_private.json'),'fresh_sha256':FRESH_TEST_SHA256}



def audit_stage(stage, paths):
    from datetime import datetime, timezone
    target=WORK/'stage_audits'; target.mkdir(exist_ok=True)
    files=[Path(p) for p in paths if Path(p).is_file()]
    for p in files: assert p.resolve().is_relative_to(WORK.resolve())
    manifest={str(p.relative_to(WORK)):file_sha(p) for p in files}
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    dest=target/f'{stage}_{stamp}_PRIVATE.zip'; temp=dest.with_suffix('.tmp')
    import zipfile
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as z:
        for p in files: z.write(p,str(p.relative_to(WORK)))
        z.writestr('audit_manifest.json',json.dumps(manifest,indent=2))
    with zipfile.ZipFile(temp) as z:
        assert z.testzip() is None
        assert all(hashlib.sha256(z.read(k)).hexdigest()==v for k,v in manifest.items())
    temp.replace(dest); print('Private stage audit:',dest)

ns=cpu('RECOVER',['approve']); ns['load_approved']()
gpu('SMOKE')
# Only an exact matching pass can authorize the next training worker.
print('NEXT: notebook 04. Keep the same GPU and training configuration.')
audit_stage(STAGE,[WORK/'project_state.json',WORK/'auto_gpu_versions.json',*[p for p in (WORK/'sft_output').glob('*/smoke_pass.json')]])